# screamingface · YAML quickstart

Keep a fusion lineup in a small, reviewable YAML file, load it without making model calls, and
run the same evaluation flow as the main quickstart.

This committed copy runs in mock mode, fully offline. For a live run, replace the setup call with
`sf.setup()` and put exact IDs from your live `sf.models.list()` into `fusion.yaml` — see the
going-live checklist in [`00_quickstart.ipynb`](00_quickstart.ipynb).

Model IDs are exact, never aliases: `hf/...` is not expanded to `huggingface/...`.

## 1 · Start a reproducible session

In [1]:
import screamingface as sf

# REMOVE mode and static_widgets to run live: sf.setup()
session = sf.setup(mode="mock", static_widgets=True)
session

SetupPanel(state='connected', credentials=<never stored>)

## 2 · Check the exact model IDs

In [2]:
available = sf.models.list()
available

['codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6']

## 3 · Review `fusion.yaml`

The file sits beside this notebook; loading it runs nothing.

```yaml
name: yaml-trio
models:
  - codex/gpt-5.5
  - gemini-cli/gemini-2.5-pro
  - anthropic/claude-sonnet-4-6
reduce: majority_vote
judge: codex/gpt-5.5
```

## 4 · Load the fusion and inspect its lineup

In [3]:
fusion = sf.Fusion.from_yaml("fusion.yaml")
fusion  # rich lineup table; the URL4 stays hidden

Role,Model
Judge,codex/gpt-5.5
Member,gemini-cli/gemini-2.5-pro
Member,anthropic/claude-sonnet-4-6


Loading a fusion needs no connected providers; live evaluation validates them up
front and fails fast with `FusionNotReady` before any call.

Ask for the shareable URL4 recipe when you need it:

In [4]:
fusion.url4

"(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=yaml-trio;sf_judge=codex/gpt-5.5"

The same schema can be supplied as an inline Python mapping:

In [5]:
fusion_config = {
    "name": "yaml-trio",
    "models": available[:3],
    "reduce": "majority_vote",
    "judge": available[0],
}
inline_fusion = sf.Fusion(**fusion_config)
inline_fusion.url4 == fusion.url4

True

## 5 · Evaluate and read the result

In [6]:
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=yaml-trio;sf_judge=codex/gpt-5.5", sample_size=20, seed=0, score=100.0, baseline=80.0, gain=20.0, cost_usd=0.0, fusion_name='yaml-trio', reduce='majority_vote', judge='codex/gpt-5.5', incomplete=0, profiles=(), pricing_source='estimate:SDK catalog', pricing_as_of='2026-07-16', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='gemini-cli/gemini-2.5-pro', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='anthropic/claude-sonnet-4-6', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0)), failures=())

Read `gain` first: positive means the fusion beat its strongest member on the same
answers.

In [7]:
{
    "score": run.score,
    "baseline": run.baseline,
    "gain": run.gain,
    "mode": run.mode,
}

{'score': 100.0, 'baseline': 80.0, 'gain': 20.0, 'mode': 'mock'}

**Next:** [`00_quickstart.ipynb`](00_quickstart.ipynb) — the same flow with an
inline Python lineup.